# BACI HS92 — Data Preparation

Pipeline: load reference files → combine all yearly CSVs → save merged dataset → build edge lists.

**Output 1:** `baci_combined.parquet` — full merged dataset (all years, all products)  
**Output 2:** `baci_edges_country.parquet` — country-level edge list → input for `baci_network_analysis.ipynb`  
**Output 3:** `baci_edges_product.parquet` — product-level edge list (stored only)

**Data:** BACI HS92 V202601 — one CSV per year, columns `t i j k v q`  
**Units:** `v` = USD thousands · `q` = metric tons

## Imports and Configuration

In [9]:

import glob
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import pyarrow



In [ ]:
# ── Configure your data path here ─────────────────────────────────────────
DATA_BACI = Path("/Users/olivia_peter/Masters BSE/Term 3/Thesis BSE/Code Thesis/BACI_HS92_V202601")  # <-- edit this
OUT_BACI  = DATA_BACI                             # parquets saved alongside the CSVs

assert DATA_BACI.exists(), f"DATA_BACI not found: {DATA_BACI}"
print("DATA_BACI:", DATA_BACI)

## Step 1 — Load Reference Files

In [11]:
# Load country codes reference table
country_codes = pd.read_csv(DATA_BACI / "country_codes_V202601.csv")
print("country_codes — shape:", country_codes.shape)
print("Columns:", country_codes.columns.tolist())
country_codes.head()


country_codes — shape: (238, 4)
Columns: ['country_code', 'country_name', 'country_iso2', 'country_iso3']


,country_code,country_name,country_iso2,country_iso3
0,4,Afghanistan,AF,AFG
1,8,Albania,AL,ALB
2,12,Algeria,DZ,DZA
3,16,American Samoa,AS,ASM
4,20,Andorra,AD,AND


In [12]:
# Load HS92 product codes — keep code column as string (leading zeros)
product_codes = pd.read_csv(
    DATA_BACI / "product_codes_HS92_V202601.csv",
    dtype={"code": str},
)
print("product_codes — shape:", product_codes.shape)
print("Columns:", product_codes.columns.tolist())
product_codes.head()


product_codes — shape: (5022, 2)
Columns: ['code', 'description']


,code,description
0,010111,"Horses: live, pure-bred breeding animals"
1,010119,"Horses: live, other than pure-bred breeding an..."
2,010120,"Asses, mules and hinnies: live"
3,010210,"Bovine animals: live, pure-bred breeding animals"
4,010290,"Bovine animals: live, other than pure-bred bre..."


In [14]:
# ── Confirm and normalise country code format ─────────────────────────────
# Identify column names (adjust if your file differs)
CODE_COL = "country_code"   # numeric country identifier in country_codes
ISO3_COL = "country_iso3"   # ISO-3 alpha code in country_codes

print("--- country_codes ---")
print(f"  '{CODE_COL}' dtype  :", country_codes[CODE_COL].dtype)
print(f"  sample values       :", country_codes[CODE_COL].head(5).tolist())

# Peek at the first yearly CSV to check 'i' format
first_csv = sorted(DATA_BACI.glob("BACI_HS92_Y*_V202601.csv"))[0]
df_peek = pd.read_csv(first_csv, nrows=5, dtype={"k": str})
print("\n--- first yearly CSV ---")
print(f"  'i' dtype           :", df_peek["i"].dtype)
print(f"  sample values       :", df_peek["i"].head(5).tolist())
del df_peek

# Normalise both to plain int32 so merges are unambiguous
if country_codes[CODE_COL].dtype == object:
    country_codes[CODE_COL] = (
        pd.to_numeric(country_codes[CODE_COL], errors="coerce")
        .astype("Int32")
    )
    print("\nFix applied: country_codes[CODE_COL] converted from str → Int32")
else:
    country_codes[CODE_COL] = country_codes[CODE_COL].astype("Int32")
    print("\nNo string-to-int fix needed — already numeric; cast to Int32")

print("\nFinal country_codes dtypes:")
print(country_codes.dtypes)


--- country_codes ---
  'country_code' dtype  : int64
  sample values       : [4, 8, 12, 16, 20]

--- first yearly CSV ---
  'i' dtype           : int64
  sample values       : [4, 4, 4, 4, 4]

No string-to-int fix needed — already numeric; cast to Int32

Final country_codes dtypes:
country_code     Int32
country_name    object
country_iso2    object
country_iso3    object
dtype: object


## Step 2 — Load and Combine All Yearly CSVs

In [16]:
# ── Discover yearly files ─────────────────────────────────────────────────
csv_files = sorted(DATA_BACI.glob("BACI_HS92_Y*_V202601.csv"))
print(f"Found {len(csv_files)} yearly CSV files:")
for f in csv_files:
    print(f"  {f.name}")


Found 30 yearly CSV files:
  BACI_HS92_Y1995_V202601.csv
  BACI_HS92_Y1996_V202601.csv
  BACI_HS92_Y1997_V202601.csv
  BACI_HS92_Y1998_V202601.csv
  BACI_HS92_Y1999_V202601.csv
  BACI_HS92_Y2000_V202601.csv
  BACI_HS92_Y2001_V202601.csv
  BACI_HS92_Y2002_V202601.csv
  BACI_HS92_Y2003_V202601.csv
  BACI_HS92_Y2004_V202601.csv
  BACI_HS92_Y2005_V202601.csv
  BACI_HS92_Y2006_V202601.csv
  BACI_HS92_Y2007_V202601.csv
  BACI_HS92_Y2008_V202601.csv
  BACI_HS92_Y2009_V202601.csv
  BACI_HS92_Y2010_V202601.csv
  BACI_HS92_Y2011_V202601.csv
  BACI_HS92_Y2012_V202601.csv
  BACI_HS92_Y2013_V202601.csv
  BACI_HS92_Y2014_V202601.csv
  BACI_HS92_Y2015_V202601.csv
  BACI_HS92_Y2016_V202601.csv
  BACI_HS92_Y2017_V202601.csv
  BACI_HS92_Y2018_V202601.csv
  BACI_HS92_Y2019_V202601.csv
  BACI_HS92_Y2020_V202601.csv
  BACI_HS92_Y2021_V202601.csv
  BACI_HS92_Y2022_V202601.csv
  BACI_HS92_Y2023_V202601.csv
  BACI_HS92_Y2024_V202601.csv


In [18]:
# ── Per-file dtypes — k MUST be str to preserve leading zeros ─────────────
LOAD_DTYPES = {
    "t": "int16",    # year fits in int16 (max 32767)
    "i": "int32",    # exporter country code
    "j": "int32",    # importer country code
    "k": str,        # HS6 product code — mandatory string
    "v": "float32",  # trade value (USD thousands)
    "q": "float32",  # quantity (metric tons)
}

# ── Pass 1: CSV → yearly parquets (idempotent — safe to re-run) ───────────
for fp in csv_files:
    year = fp.stem.split("_Y")[1].split("_")[0]
    out_parq = OUT_BACI / f"baci_{year}.parquet"
    if out_parq.exists():
        print(f"  {out_parq.name} already exists — skipped")
        continue
    df = pd.read_csv(fp, dtype=LOAD_DTYPES)
    df.to_parquet(out_parq, engine="pyarrow", compression="snappy", index=False)
    del df
    print(f"  {fp.name} → {out_parq.name}")

# ── Pass 2: PyArrow-level concat (no peak-RAM double-load) ────────────────
yearly_parquets = sorted(OUT_BACI.glob("baci_[0-9]*.parquet"))
print(f"\nLoading {len(yearly_parquets)} yearly parquets...")
baci_1995_2024 = pd.read_parquet(yearly_parquets, engine="pyarrow")

print(f"\nCombined shape: {baci_1995_2024.shape}")
print(f"Memory (raw):   {baci_1995_2024.memory_usage(deep=True).sum() / 1e9:.2f} GB")

  baci_1995.parquet already exists — skipped
  baci_1996.parquet already exists — skipped
  baci_1997.parquet already exists — skipped
  baci_1998.parquet already exists — skipped
  baci_1999.parquet already exists — skipped
  baci_2000.parquet already exists — skipped
  baci_2001.parquet already exists — skipped
  baci_2002.parquet already exists — skipped
  baci_2003.parquet already exists — skipped
  baci_2004.parquet already exists — skipped
  baci_2005.parquet already exists — skipped
  baci_2006.parquet already exists — skipped
  baci_2007.parquet already exists — skipped
  baci_2008.parquet already exists — skipped
  baci_2009.parquet already exists — skipped
  baci_2010.parquet already exists — skipped
  baci_2011.parquet already exists — skipped
  baci_2012.parquet already exists — skipped
  baci_2013.parquet already exists — skipped
  baci_2014.parquet already exists — skipped
  baci_2015.parquet already exists — skipped
  baci_2016.parquet already exists — skipped
  baci_201

In [ ]:
# ── Drop invalid rows (idempotent — safe to re-run) ───────────────────────
n_before = len(baci_1995_2024)

# 1. Null or non-positive value
baci_1995_2024 = baci_1995_2024[baci_1995_2024["v"].notna() & (baci_1995_2024["v"] > 0)]

# 2. Self-loops (exporter == importer) — only if i/j still present
if {"i", "j"}.issubset(baci_1995_2024.columns):
    baci_1995_2024 = baci_1995_2024[baci_1995_2024["i"] != baci_1995_2024["j"]]
elif {"iso3_exporter", "iso3_importer"}.issubset(baci_1995_2024.columns):
    baci_1995_2024 = baci_1995_2024[
        baci_1995_2024["iso3_exporter"] != baci_1995_2024["iso3_importer"]
    ]

n_after = len(baci_1995_2024)
print(f"Rows before cleaning : {n_before:>12,}")
print(f"Rows after cleaning  : {n_after:>12,}")
print(f"Rows dropped         : {n_before - n_after:>12,}")

In [ ]:
# ── Map numeric codes → ISO3 strings (idempotent — safe to re-run) ───────
if "iso3_exporter" in baci_1995_2024.columns:
    print("Already mapped to ISO3 — skipping.")
else:
    code_to_iso3 = country_codes.set_index(CODE_COL)[ISO3_COL].to_dict()

    baci_1995_2024 = baci_1995_2024.copy()
    baci_1995_2024["iso3_exporter"] = baci_1995_2024["i"].map(code_to_iso3)
    baci_1995_2024["iso3_importer"] = baci_1995_2024["j"].map(code_to_iso3)

    # Drop rows where mapping failed (non-standard BACI country codes)
    unmapped = baci_1995_2024["iso3_exporter"].isna() | baci_1995_2024["iso3_importer"].isna()
    print(f"Rows unmapped (dropped): {unmapped.sum():,}")
    baci_1995_2024 = baci_1995_2024.loc[~unmapped].copy()
    print(f"Rows remaining         : {len(baci_1995_2024):,}")

    # Convert to memory-efficient category dtype
    baci_1995_2024["iso3_exporter"] = baci_1995_2024["iso3_exporter"].astype("category")
    baci_1995_2024["iso3_importer"] = baci_1995_2024["iso3_importer"].astype("category")

    # Drop now-redundant integer code columns
    baci_1995_2024 = baci_1995_2024.drop(columns=["i", "j"])

In [ ]:
# ── Add HS2 chapter column ────────────────────────────────────────────────
baci_1995_2024["hs2"] = baci_1995_2024["k"].str[:2].astype("category")

print("Column dtypes after enrichment:")
print(baci_1995_2024.dtypes)
print(f"\nMemory usage: {baci_1995_2024.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# ── Save combined dataset to parquet ─────────────────────────────────────
out_path = OUT_BACI / "baci_combined.parquet"
mem_before = baci_1995_2024.memory_usage(deep=True).sum() / 1e9

baci_1995_2024.to_parquet(out_path, engine="pyarrow", index=False, compression="snappy")

file_size_gb = out_path.stat().st_size / 1e9
print(f"Saved → {out_path.name}")
print(f"Memory in RAM  : {mem_before:.2f} GB")
print(f"File on disk   : {file_size_gb:.2f} GB")
print(f"Compression    : {mem_before / file_size_gb:.1f}x")

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────
print("=" * 55)
print("BACI COMBINED — SUMMARY")
print("=" * 55)

years = sorted(baci_1995_2024["t"].unique())
print(f"Year range        : {years[0]} – {years[-1]}  ({len(years)} years)")
print(f"Total rows        : {len(baci_1995_2024):,}")
print(f"Unique exporters  : {baci_1995_2024['iso3_exporter'].nunique()}")
print(f"Unique importers  : {baci_1995_2024['iso3_importer'].nunique()}")
print(f"Unique HS6 codes  : {baci_1995_2024['k'].nunique()}")
print(f"Unique HS2 chaps  : {baci_1995_2024['hs2'].nunique()}")

print("\nTotal trade value by year (USD trillions):")
yr_totals = baci_1995_2024.groupby("t")["v"].sum() / 1e9  # USD thousands → trillions
for yr, val in yr_totals.items():
    print(f"  {yr}: {val:8.3f} T USD")

print("\nTop 5 exporters (all years, USD trillions):")
top_exp = (
    baci_1995_2024.groupby("iso3_exporter")["v"].sum() / 1e9
).nlargest(5)
for iso, val in top_exp.items():
    print(f"  {iso}: {val:8.3f}")

print("\nTop 5 importers (all years, USD trillions):")
top_imp = (
    baci_1995_2024.groupby("iso3_importer")["v"].sum() / 1e9
).nlargest(5)
for iso, val in top_imp.items():
    print(f"  {iso}: {val:8.3f}")

## Step 3 — Build Edge Lists

### 3a — Product-Level Edge List (full granularity, store only)

In [ ]:
# Rename columns to semantic names and save.
# This file is written once and NOT kept in memory — use for future analysis only.
product_edge_path = OUT_BACI / "baci_edges_product.parquet"

(
    baci_1995_2024
    .rename(columns={
        "t": "year",
        "iso3_exporter": "source",
        "iso3_importer": "target",
        "k": "hs6",
        "hs2": "hs2",
        "v": "value",    # USD thousands
        "q": "quantity"  # metric tons
    })
    .to_parquet(product_edge_path, engine="pyarrow", index=False, compression="snappy")
)

size_gb = product_edge_path.stat().st_size / 1e9
print(f"Saved → {product_edge_path.name}")
print(f"File size on disk : {size_gb:.3f} GB")
print("(File kept on disk only — not loaded into memory)")

### 3b — Country-Level Edge List (for all subsequent analysis)

In [ ]:
# Aggregate across all products: sum value per (year, exporter, importer)
country_edges = (
    baci_1995_2024
    .groupby(["t", "iso3_exporter", "iso3_importer"], observed=True)["v"]
    .sum()
    .reset_index()
)

# Convert USD thousands → USD billions
country_edges["v"] = country_edges["v"] / 1_000_000

# Rename to clear semantic names
country_edges = country_edges.rename(columns={
    "t":              "year",
    "iso3_exporter":  "source",
    "iso3_importer":  "target",
    "v":              "value",   # USD billions
})

# Reset category dtype after groupby (categories may have stale levels)
country_edges["source"] = country_edges["source"].astype("category")
country_edges["target"] = country_edges["target"].astype("category")

# Save
country_edge_path = OUT_BACI / "baci_edges_country.parquet"
country_edges.to_parquet(country_edge_path, engine="pyarrow", index=False, compression="snappy")

print(f"Saved → {country_edge_path.name}")
print(f"Shape        : {country_edges.shape}")
yr = country_edges['year']
print(f"Year range   : {yr.min()} – {yr.max()}")
print(f"\nEdge weight (USD billions):")
print(f"  min    : {country_edges['value'].min():.6f}")
print(f"  median : {country_edges['value'].median():.4f}")
print(f"  mean   : {country_edges['value'].mean():.4f}")
print(f"  max    : {country_edges['value'].max():.2f}")